# Capstone — mirrors your deployed research paper

**Lane 2 · Refresh / Content Opportunity Scoring · Google Search Ranking & Discoverability**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnanyaGujral/flyrank-ml-internship-start/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook is the **executable mirror** of the deployed paper. It runs top to bottom **offline** on the
bundled anonymized slice (`data/raw/content_refresh_anonymized.csv`, 30,000 pages), reads the Week-3→7
receipts in `work/outputs/`, and regenerates every figure and summary the paper embeds.

## 1. Question

**Decision this supports:** a FlyRank content editor has a weekly budget of a few hundred page reviews
across a portfolio whose libraries hold thousands of pages. Which already-visible pages should be
reviewed **first**?

**Research question.** *Among a client portfolio's already-visible pages, can trailing search and
engagement signals rank the pages that are about to decline well enough to beat a transparent rule —
measured at the top of the queue, under a client-grouped (new-client) validation design?*

In [1]:
# ---- capstone role bootstrap ----
import json
from pathlib import Path

root = Path.cwd().resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
OUT = root / "work" / "outputs"          # committed receipts
ART = root / "output"                    # paper-facing artifacts
PORT = root / "portfolio" / "assets"     # files the deployed paper embeds
for d in (ART / "charts", PORT):
    d.mkdir(parents=True, exist_ok=True)
receipts = {p.stem: json.loads(p.read_text(encoding="utf-8")) for p in sorted(OUT.glob("*.json"))}
print("receipts found:", ", ".join(sorted(receipts)))

receipts found: w03_data_contract_metrics, w03_feature_leakage_metrics, w04_baseline_metrics, w04_signal_audit_metrics, w05_model_metrics, w06_validation_metrics, w07_playbook_metrics


## 2. Data

One row = one pseudonymized page; 30,000 pages across 32 clients; a single trailing-90-day snapshot that
also stores two stacked 30-day windows. **Features are read from the prior-30 window (days 31–60);
the label is read from the later-30 window (days 1–30).** Nothing from the 90-day totals or the outcome
window enters the feature matrix.

In [2]:
# ---- shared offline setup: bundled slice, leak-free pool + features, transparent rule ----
import json
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
MIN_PREV_IMPRESSIONS = 100     # "already visible" floor on the TRAILING window
DROP_FACTOR = 0.8              # declined = next-30 impressions < 80% of prior-30

root = Path.cwd().resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
DATA = root / "data" / "raw" / "content_refresh_anonymized.csv"
OUT = root / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(DATA)


def build_pool(frame):
    """Already-visible pages with a usable trailing window, plus leak-free features.

    Every feature is measured in the PRIOR 30-day window (days 31-60) or is static
    metadata. Nothing is taken from the 90-day totals or the outcome window, because
    those overlap the label's comparison window.
    """
    pool = frame[frame["impressions_prev_30d"] >= MIN_PREV_IMPRESSIONS].copy()
    pool["declined_next_30d"] = (
        pool["impressions_last_30d"] < DROP_FACTOR * pool["impressions_prev_30d"]).astype(int)
    pool["log_impr_prev"] = np.log1p(pool["impressions_prev_30d"])
    pool["log_clk_prev"] = np.log1p(pool["clicks_prev_30d"])
    pool["log_sess_prev"] = np.log1p(pool["sessions_prev_30d"])
    pool["ctr_prev"] = 100 * pool["clicks_prev_30d"] / pool["impressions_prev_30d"]
    pool["clk_per_sess_prev"] = pool["clicks_prev_30d"] / pool["sessions_prev_30d"].replace(0, np.nan)
    pool["has_keyword"] = pool["search_volume"].notna().astype(int)
    pool["has_clicks_prev"] = (pool["clicks_prev_30d"] > 0).astype(int)
    pool["has_sessions_prev"] = (pool["sessions_prev_30d"] > 0).astype(int)
    g = pool.groupby("client_id")
    pool["ctr_prev_z"] = ((pool["ctr_prev"] - g["ctr_prev"].transform("mean"))
                          / g["ctr_prev"].transform("std").replace(0, np.nan))
    pool["impr_prev_share"] = pool["impressions_prev_30d"] / g["impressions_prev_30d"].transform("sum")
    pool["clk_prev_share"] = pool["clicks_prev_30d"] / g["clicks_prev_30d"].transform("sum").replace(0, np.nan)
    return pool


NUM_FEATURES = ["log_impr_prev", "log_clk_prev", "log_sess_prev", "ctr_prev", "clk_per_sess_prev",
                "ctr_prev_z", "impr_prev_share", "clk_prev_share",
                "search_volume", "competition", "cpc", "word_count", "char_count",
                "content_age_days", "days_since_last_update",
                "has_keyword", "has_clicks_prev", "has_sessions_prev"]
CAT_FEATURES = ["competition_level", "content_type", "main_intent", "age_tier",
                "freshness_tier", "word_count_tier"]


def feature_matrix(pool):
    Xn = pool[NUM_FEATURES].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return pd.concat([Xn, pd.get_dummies(pool[CAT_FEATURES].astype(str), dtype=float)], axis=1)


def rulescore(pool):
    """Transparent rule: low trailing CTR, weighted by trailing exposure."""
    med = float(pool["ctr_prev"].median())
    return (pool["ctr_prev"] <= 0.5 * med).to_numpy() * np.log1p(pool["impressions_prev_30d"].to_numpy())


pool = build_pool(raw)
X = feature_matrix(pool)
y = pool["declined_next_30d"].to_numpy()
groups = pool["client_id"].to_numpy()
rule = rulescore(pool)
print(f"bundled slice: {len(raw):,} rows -> pool {len(pool):,} pages | "
      f"{pool['client_id'].nunique()} pseudo-clients")
print(f"decline base rate in pool: {y.mean():.3f} | feature matrix: {X.shape}")
print("label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)")


def pk(table_scores, y_true, ks=(10, 20, 50, 100)):
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(table_scores))
    return {str(k): round(float(y_true[order[:k]].mean()), 3) for k in ks}

bundled slice: 30,000 rows -> pool 18,010 pages | 30 pseudo-clients
decline base rate in pool: 0.616 | feature matrix: (18010, 43)
label: next-30 impressions < 80% of prior-30 (all features from the prior-30 window + metadata)


## 3. Methodology

- **Pool:** pages with at least 100 prior-30 impressions (already visible).
- **Label:** `declined_next_30d = impressions_last_30d < 0.8 × impressions_prev_30d`.
- **Features:** prior-window volume/CTR/session signals (raw + log + per-client normalised), keyword
  context, depth, age/freshness, presence flags, categorical descriptors.
- **Baseline:** a transparent rule — low prior CTR (≤ half the pool median) weighted by prior exposure.
- **Models:** logistic regression, random forest, histogram gradient boosting.
- **Validation:** whole clients held out (30%, seed 42), plus GroupKFold(5) for stability, plus a
  random-split and a leak-trap contrast to show what invalid designs would report.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED).split(X, y, groups))
print(f"pool {len(pool):,} pages / {pool['client_id'].nunique()} clients | base rate {y.mean():.3f}")
print(f"held-out: {len(te):,} pages / {pd.Series(groups[te]).nunique()} clients | base {y[te].mean():.3f}")
print(f"rule precision@K (whole pool): {pk(rule, y)}")
print(f"rule precision@K (held-out)  : {pk(rule[te], y[te])}")

pool 18,010 pages / 30 clients | base rate 0.616
held-out: 3,571 pages / 9 clients | base 0.567
rule precision@K (whole pool): {'10': 0.6, '20': 0.75, '50': 0.8, '100': 0.78}
rule precision@K (held-out)  : {'10': 0.7, '20': 0.7, '50': 0.76, '100': 0.67}


## 4. Results (vs baseline)

The headline comparison comes from the Week-5 receipt (`w05_model_metrics.json`); the validation
contrasts come from Week 6 (`w06_validation_metrics.json`).

In [4]:
w05 = receipts["w05_model_metrics"]
w06 = receipts["w06_validation_metrics"]
w04 = receipts["w04_baseline_metrics"]

ks = ["10", "20", "50", "100"]
print(f"held-out base rate: {w05['test_base_rate']:.3f} | test rows {w05['test_rows']:,}\n")
print(f"{'method':<26}" + "".join(f" P@{k:<5}" for k in ks) + "  ROC-AUC")
for name, row in w05["table"].items():
    print(f"{name:<26}" + "".join(f" {row['precision_at_k'][k]:<7.3f}" for k in ks) + f"  {row['roc_auc']}")
print(f"\nGroupKFold(5) P@50 - random forest: {w06['section3_groupkfold5']['p50_mean']:.3f} "
      f"+/- {w06['section3_groupkfold5']['p50_sd']:.3f}")
print(f"GroupKFold(5) P@50 - transparent rule: {w06['section3_rule_groupkfold5']['p50_mean']:.3f} "
      f"+/- {w06['section3_rule_groupkfold5']['p50_sd']:.3f}")
print(f"\nvalidation inflation (forest): random split P@50 {w06['section2_random_split']['p50']} "
      f"(clients leaked {w06['section2_random_split']['clients_leaked']}) -> "
      f"grouped split P@50 {w06['section2_grouped_split']['p50']} (clients leaked 0)")
print(f"leak trap (+ outcome window): P@50 {w06['section3_leak_trap']['p50']} | "
      f"AUC {w06['section3_leak_trap']['auc']}")

held-out base rate: 0.567 | test rows 3,571

method                     P@10    P@20    P@50    P@100    ROC-AUC
transparent rule           0.700   0.700   0.760   0.670    0.58
logistic regression        0.700   0.750   0.760   0.740    0.585
random forest              0.500   0.500   0.640   0.600    0.582
hist gradient boosting     0.400   0.550   0.580   0.590    0.582
rule + best model blend    0.700   0.750   0.780   0.750    0.61

GroupKFold(5) P@50 - random forest: 0.788 +/- 0.148
GroupKFold(5) P@50 - transparent rule: 0.748 +/- 0.126

validation inflation (forest): random split P@50 0.96 (clients leaked 27) -> grouped split P@50 0.64 (clients leaked 0)
leak trap (+ outcome window): P@50 1.0 | AUC 0.864


## 5. Limitations

Observed, measured, directional, decision-support — never causal, never "predicting Google".

In [5]:
audit = [
    ("causal refresh effect", "NOT CLAIMED - needs a controlled experiment"),
    ("Google's ranking algorithm", "NOT CLAIMED - only observed impressions/clicks/position/engagement"),
    ("client-identifying detail", "NOT CLAIMED - pseudonymous ids only; no names/URLs/queries"),
    ("model beats the rule", "NOT SHOWN - rule and forest are within fold spread on this snapshot"),
    ("generalises beyond one snapshot", "NOT SHOWN - single 90-day slice, one bi-monthly window pair"),
]
for claim, status in audit:
    print(f"[{status}] {claim}")
print("\nobserved: decline is the majority outcome in the visible pool")
print("measured: rule vs model ranking quality on whole held-out clients")

[NOT CLAIMED - needs a controlled experiment] causal refresh effect
[NOT CLAIMED - only observed impressions/clicks/position/engagement] Google's ranking algorithm
[NOT CLAIMED - pseudonymous ids only; no names/URLs/queries] client-identifying detail
[NOT SHOWN - rule and forest are within fold spread on this snapshot] model beats the rule
[NOT SHOWN - single 90-day slice, one bi-monthly window pair] generalises beyond one snapshot

observed: decline is the majority outcome in the visible pool
measured: rule vs model ranking quality on whole held-out clients


## 6. Ranked recommendations

A four-tier, human-reviewed queue. The tiering is decision-support; the *measured* ranking quality is
only the held-out-client precision@K reported above.

In [6]:
w07 = receipts["w07_playbook_metrics"]
mix = w07["decision_support_queue"]["action_mix"]
reason = w07["top_reason_codes_review_first"]
print("action mix:", mix)
print("top reason codes in review-first:", reason)
print(f"review-first tier carries {100*w07['decision_support_queue']['review_first_share_of_demand']:.1f}% "
      f"of pool impressions")

med_ctr = float(pool["ctr_prev"].median())

def reason_codes(i):
    out = []
    if pool["ctr_prev"].iloc[i] <= 0.5 * med_ctr:
        out.append("ctr_gap")
    if pool["clicks_prev_30d"].iloc[i] == 0 and pool["impressions_prev_30d"].iloc[i] >= 500:
        out.append("no_clicks")
    if pool["impressions_prev_30d"].iloc[i] >= pool["impressions_prev_30d"].quantile(0.75):
        out.append("high_exposure")
    return "+".join(out) if out else "base_anchor"

queue = pool.copy()
queue["score"] = rule
queue["reason_code"] = [reason_codes(i) for i in range(len(queue))]
pct = queue["score"].rank(pct=True, ascending=False)
queue["action"] = np.select([pct <= 0.10, pct <= 0.25, pct <= 0.50],
                            ["investigate_first", "watch_prepare", "monitor"], default="no_action")
queue = queue.sort_values("score", ascending=False).reset_index(drop=True)

lines = ["# Ranked recommendations — the action playbook", "",
         "Decision-support ranking of already-visible pages for human review. "
         "Ranking quality is measured on whole held-out clients; tiering is a review convention.", "",
         f"- Pool: **{len(queue):,} pages** across {queue['client_id'].nunique()} pseudonymized clients",
         f"- Whole-pool decline base rate: {y.mean():.3f}",
         f"- Held-out clients: rule P@50 {w04['precision_at_k_heldout_clients']['50']}, "
         f"forest P@50 {w05['table']['random forest']['precision_at_k']['50']} (base {w05['test_base_rate']})",
         "", "| action | pages | what it means |", "|---|---:|---|",
         f"| investigate_first | {mix['investigate_first']:,} | top 10% — editor opens these this week |",
         f"| watch_prepare | {mix['watch_prepare']:,} | next 15% — queue behind the top tier |",
         f"| monitor | {mix['monitor']:,} | next 25% — light check, no rewrite |",
         f"| no_action | {mix['no_action']:,} | healthy or out of scope |",
         "", "## Top reason codes (review-first tier)", ""]
for code, n in reason.items():
    lines.append(f"- `{code}` — {n:,} pages")
lines += ["", "## Limits", "",
          "- One snapshot; the label is a window comparison, so seasonal pages can look like decliners.",
          "- Scores are rankings, not calibrated probabilities.",
          "- Never automate the action: the queue orders a human review (see `w07_action_playbook.ipynb`)."]
(ART / "ranked_recommendations.md").write_text("\n".join(lines), encoding="utf-8")
print("wrote output/ranked_recommendations.md")

action mix: {'investigate_first': 1801, 'watch_prepare': 2709, 'monitor': 3127, 'no_action': 10373}
top reason codes in review-first: {'ctr_gap+no_clicks': 881, 'ctr_gap+high_exposure': 577, 'ctr_gap+no_clicks+high_exposure': 191, 'ctr_gap': 152}
review-first tier carries 15.7% of pool impressions


wrote output/ranked_recommendations.md


## 7. Artifacts the paper embeds

Everything the deployed paper shows is regenerated here from the receipts — figures in `output/charts/`
(mirrored into `portfolio/assets/`), summaries in `output/*.md`, and the machine-readable
`output/capstone_results.json`.

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": 0.3})
CH = ART / "charts"

# 1. precision@K: rule vs models
methods = ["transparent rule", "logistic regression", "random forest"]
colors = ["#9aa0a6", "#4285F4", "#d93025"]
fig, ax = plt.subplots(figsize=(7, 3.8))
width = 0.25
for j, (m, c) in enumerate(zip(methods, colors)):
    row = w05["table"][m]["precision_at_k"]
    vals = [row[k] for k in ks]
    ax.bar(np.arange(len(ks)) + (j - 1) * width, vals, width, label=m, color=c)
ax.axhline(w05["test_base_rate"], color="black", ls="--", lw=1, label=f"base rate {w05['test_base_rate']}")
ax.set_xticks(range(len(ks))); ax.set_xticklabels([f"P@{k}" for k in ks])
ax.set_ylabel("precision"); ax.set_title("Held-out clients: does the model beat the rule?")
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(CH / "precision_at_k.svg"); plt.close(fig)

# 2. validation inflation
fig, ax = plt.subplots(figsize=(6, 3.4))
labels = ["random split\n(clients leaked)", "grouped split\n(new clients)"]
vals = [w06["section2_random_split"]["p50"], w06["section2_grouped_split"]["p50"]]
bars = ax.bar(labels, vals, color=["#d93025", "#188038"])
ax.axhline(w05["test_base_rate"], color="black", ls="--", lw=1, label=f"base rate {w05['test_base_rate']}")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("forest P@50"); ax.set_title("Wrong split flatters the model")
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(CH / "validation_inflation.svg"); plt.close(fig)

# 3. action mix
fig, ax = plt.subplots(figsize=(6.5, 3.5))
order = ["investigate_first", "watch_prepare", "monitor", "no_action"]
vals = [100 * mix[a] / sum(mix.values()) for a in order]
bars = ax.bar(order, vals, color=["#d93025", "#f9ab00", "#4285F4", "#9aa0a6"])
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("% of pool"); ax.set_title("The four actions and their share of the pool")
fig.tight_layout(); fig.savefig(CH / "action_mix.svg"); plt.close(fig)

# 4. signal lift (from the Week-4 audit)
fig, ax = plt.subplots(figsize=(7, 3.6))
flags = receipts["w04_signal_audit_metrics"]["flag_lift"]
names = [f["flag"] for f in flags]; lifts = [f["lift"] for f in flags]
ax.barh(names, lifts, color=["#188038" if v > 0.02 else "#9aa0a6" for v in lifts])
ax.axvline(0, color="black", lw=1); ax.set_xlabel("decline-rate lift vs not-flagged")
ax.set_title("Which trailing flags actually separate decliners?")
fig.tight_layout(); fig.savefig(CH / "signal_lift.svg"); plt.close(fig)
print("charts:", ", ".join(sorted(p.name for p in CH.glob('*.svg'))))

import shutil
for p in CH.glob("*.svg"):
    shutil.copyfile(p, PORT / p.name)

# summaries
runtime = pool[[c for c in ["ctr_prev"] if c in pool]].describe().T
(ART / "evaluation_summary.md").write_text("\n".join([
    "# Evaluation summary", "",
    f"Offline study on the bundled anonymized slice — `data/raw/content_refresh_anonymized.csv` "
    f"({len(raw):,} pages, {raw['client_id'].nunique()} clients). Pool {len(pool):,} pages. "
    f"Label: later-30 impressions < 80% of prior-30. Features: prior-30 window + metadata only.", "",
    f"Validation: whole clients held out (seed 42, 30% of clients); base rate {w05['test_base_rate']}.", "",
    "| method | P@10 | P@20 | P@50 | P@100 | ROC-AUC |", "|---|---:|---:|---:|---:|---:|",
    *[f"| {m} | " + " | ".join(f"{w05['table'][m]['precision_at_k'][k]:.3f}" for k in ks)
      + f" | {w05['table'][m]['roc_auc']:.3f} |" for m in methods], "",
    f"GroupKFold(5) P@50: forest {w06['section3_groupkfold5']['p50_mean']:.3f} ± "
    f"{w06['section3_groupkfold5']['p50_sd']:.3f}; rule {w06['section3_rule_groupkfold5']['p50_mean']:.3f} ± "
    f"{w06['section3_rule_groupkfold5']['p50_sd']:.3f}.", "",
    f"Invalid designs: random split forest P@50 {w06['section2_random_split']['p50']} with "
    f"{w06['section2_random_split']['clients_leaked']} clients leaked; leak trap P@50 "
    f"{w06['section3_leak_trap']['p50']} / AUC {w06['section3_leak_trap']['auc']}.", "",
    "**Honest reading:** the trailing features carry little signal about next-window decline; the "
    "transparent rule is at least as good as the learned models, and naive validation makes the model "
    "look far better than it is.",
]), encoding="utf-8")

(ART / "data_quality_summary.md").write_text("\n".join([
    "# Data quality summary", "",
    f"- Slice: {len(raw):,} rows × {raw.shape[1]} columns, {raw['client_id'].nunique()} pseudonymized clients.",
    f"- Grain holds: {raw['content_id'].nunique():,} unique `content_id` values (duplicates: "
    f"{int(raw['content_id'].duplicated().sum())}).",
    f"- Visibility floor `impressions_prev_30d >= 100` keeps {len(pool):,} pages "
    f"({100*len(pool)/len(raw):.1f}% of the slice).",
    f"- Rows with no prior-window denominator (`impressions_prev_30d == 0`): "
    f"{int((raw['impressions_prev_30d'] == 0).sum()):,} — excluded by the floor.",
    f"- `avg_position == 0` means \"no data\": {int((raw['avg_position'] == 0).sum()):,} rows.",
    "- Rate columns are ×100 percentages; `scroll_rate` / `ai_traffic_pct` can exceed 100 by design.",
    "- Missingness in `search_volume` / `word_count` is content-type-linked, not random.",
    "- Two stacked 30-day windows are the smallest honest decision/outcome pair this slice offers.",
]), encoding="utf-8")

(ART / "README.md").write_text("\n".join([
    "# output/ — paper-facing artifacts", "",
    "Regenerated by `work/notebooks/capstone.ipynb` (and mirrored into `portfolio/assets/`).", "",
    "| file | what it is |", "|---|---|",
    "| `evaluation_summary.md` | rule-vs-model table + validation contrasts |",
    "| `data_quality_summary.md` | grain, floor, units, missingness checks |",
    "| `ranked_recommendations.md` | the four-tier action playbook |",
    "| `capstone_results.json` | machine-readable headline numbers |",
    "| `charts/*.svg` | figures the deployed paper embeds |", "",
    "Everything derives from `work/outputs/*.json` receipts, which the notebooks write.", "",
]), encoding="utf-8")

(ART / "capstone_results.json").write_text(json.dumps({
    "study": "offline-bundled-slice",
    "data": {"rows": int(len(raw)), "clients": int(raw["client_id"].nunique()),
             "pool_rows": int(len(pool)), "pool_clients": int(pool["client_id"].nunique())},
    "label": "declined_next_30d = impressions_last_30d < 0.8 * impressions_prev_30d",
    "windows": {"features": "prior-30 (days 31-60)", "outcome": "later-30 (days 1-30)"},
    "heldout": w05["table"],
    "heldout_base_rate": w05["test_base_rate"],
    "groupkfold5": {"random_forest": w06["section3_groupkfold5"], "rule": w06["section3_rule_groupkfold5"]},
    "invalid_designs": {"random_split": w06["section2_random_split"], "leak_trap": w06["section3_leak_trap"]},
    "action_mix": mix,
    "top_reason_codes": reason,
}, indent=2), encoding="utf-8")
print("wrote output/evaluation_summary.md, data_quality_summary.md, ranked_recommendations.md, README.md, capstone_results.json")
print("mirrored charts into portfolio/assets/")

charts: action_mix.svg, feature_importance.svg, precision_at_k.svg, signal_lift.svg, split_comparison.svg, validation_inflation.svg
wrote output/evaluation_summary.md, data_quality_summary.md, ranked_recommendations.md, README.md, capstone_results.json
mirrored charts into portfolio/assets/


## Self-check

- [x] Runs top to bottom offline on the bundled slice, no token
- [x] Reads only the committed receipts in `work/outputs/` for the headline table
- [x] Heading 4 compares the model against the Week-4 baseline on the same held-out clients
- [x] Honest limitations; no causal or "Google algorithm" claim
- [x] Regenerates `output/` summaries and `output/charts/*.svg`, mirrored to `portfolio/assets/`
- [x] Fixes the random seed (42) and the client-grouped split convention
- [x] Committed under `work/notebooks/`